<a href="https://colab.research.google.com/github/deltorobarba/astrophysics/blob/main/irsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Infrared Science Archive (IRSA)**

The **IRSA Catalog** refers to the astronomical data and resources provided by the **Infrared Science Archive (IRSA)**. IRSA is an essential facility hosted by NASA, tasked with curating and distributing infrared astronomical data collected from missions and projects. It enables astronomers to search for and access large-scale datasets related to infrared observations of celestial objects.

1. **Infrared Data**: IRSA focuses on data in the infrared spectrum, which allows astronomers to study phenomena such as:
   - Star formation.
   - Interstellar dust and gas.
   - Galaxy formation and evolution.
   - Objects obscured in visible light by dust.

2. **Access to Multiple Missions**: IRSA provides access to data from several missions, such as:
   - **Spitzer Space Telescope** (e.g., GLIMPSE, SEIP catalogs).
   - **WISE (Wide-field Infrared Survey Explorer)** and its catalogs.
   - **2MASS (Two Micron All-Sky Survey)** catalog.
   - **SOFIA (Stratospheric Observatory for Infrared Astronomy)** data.
   - **Planck** and more.

3. **Data Search and Retrieval Tools**:
   - **Gator**: A catalog search engine to query and filter data based on astronomical criteria.
   - **IRSA Viewer**: A tool for visualizing infrared images and overlaying catalogs.
   - **VO (Virtual Observatory) Tools**: Interoperable tools for accessing and analyzing data.

4. **Application**:
   - Study of celestial objects (stars, galaxies, planetary systems).
   - Analysis of large-scale cosmic structures.
   - Identification of exoplanets, brown dwarfs, and asteroids.
   - Examining regions of the universe hidden in optical wavelengths.

The IRSA catalogs support astrophysical research by enabling detailed analysis of phenomena that are not accessible in visible light. For example:
- Infrared observations are crucial for understanding star formation in dense molecular clouds.
- It helps in identifying distant galaxies obscured by interstellar dust.

Researchers can use IRSA tools to retrieve photometric data, spectra, and imaging for studies ranging from nearby planetary systems to the most distant observable galaxies.

In [ ]:
!pip install astropy astroquery -q
from astroquery.ipac.irsa import Irsa
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad
from astropy import coordinates
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
import matplotlib.pyplot as plt
%matplotlib inline

https://astroquery.readthedocs.io/en/latest/ipac/irsa/irsa.html

https://astroquery.readthedocs.io/projects/keflavich-astroquery/en/latest/irsa.html

*Convert sky coordinates and double check objects*

In [ ]:
# Get skycoordinates of desired object
galaxy = Simbad.query_object('m31')
ra_hms = galaxy['RA'][0]
dec_dms = galaxy['DEC'][0]
print(galaxy)
print("RA (h:m:s):", ra_hms)
print("DEC (d:m:s):", dec_dms)

MAIN_ID      RA          DEC      RA_PREC ... COO_WAVELENGTH     COO_BIBCODE     SCRIPT_NUMBER_ID
          "h:m:s"      "d:m:s"            ...                                                    
------- ------------ ------------ ------- ... -------------- ------------------- ----------------
  M  31 00 42 44.330 +41 16 07.50       7 ...              I 2006AJ....131.1163S                1
RA (h:m:s): 00 42 44.330
DEC (d:m:s): +41 16 07.50


In [ ]:
# Convert Right Ascension (RA) and Declination (DEC) to decimal degree
coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=('hourangle', 'deg'), frame='icrs')
ra_deg = coord.ra.deg # round(coord.ra.deg, 2)
dec_deg = coord.dec.deg # round(coord.dec.deg, 2)

print("RA in degrees:", ra_deg)
print("DEC in degrees:", dec_deg)

RA in degrees: 10.684708333333331
DEC in degrees: 41.26875


In [ ]:
# Convert from decimal degree to Right Ascension (RA) and Declination (DEC)
# RA: hours, minutes, seconds. DEC: degrees, arcminutes, arcseconds
# Create a SkyCoord object
coord = SkyCoord(ra=ra_deg, dec=dec_deg, unit='deg', frame='icrs')

# Format RA and DEC with specific style
ra_hms = coord.ra.to_string(unit='hour', sep=' ', precision=3, pad=True)
dec_dms = coord.dec.to_string(unit='deg', sep=' ', precision=2, alwayssign=True, pad=True)

print("RA (h:m:s):", ra_hms)
print("DEC (d:m:s):", dec_dms)

RA (h:m:s): 00 42 44.330
DEC (d:m:s): +41 16 07.50


In [ ]:
# Reverse check to see to which object the coordinates belong
coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=('hourangle', 'deg'), frame='icrs')
result = Simbad.query_region(coord, radius='0d0m5s')  # Search within a small radius (5 arcseconds)
if result:
    print(result['MAIN_ID'][0])  # Show the main name of the object
else:
    print("No object found in SIMBAD at these coordinates.")
object_name = 'MAIN_ID'

M  31


*Query IRSA catalogue*

In [ ]:
# Option 1: Display available data by searching for the object name
table = Irsa.query_region("m31", catalog="fp_psc", spatial="Cone",
                          radius=2 * u.arcmin)
print(table)

    ra        dec     err_maj err_min err_ang ... scan_key coadd_key coadd        htm20       
   deg        deg      arcsec  arcsec   deg   ...                                             
---------- ---------- ------- ------- ------- ... -------- --------- ----- -------------------
 10.692216  41.260162    0.10    0.09      87 ...    69157   1590591    33 4805203678124326400
 10.700059  41.263481    0.31    0.30     155 ...    69157   1590591    33 4805203678125364736
 10.699131  41.263248    0.28    0.20      82 ...    69157   1590591    33 4805203678125474304
 10.697569  41.261272    0.11    0.10      90 ...    69157   1590591    33 4805203678125530624
 10.703106  41.252998    0.16    0.14      21 ...    69157   1590591    33 4805203678126695936
 10.703557  41.252811    0.14    0.13     111 ...    69157   1590591    33 4805203678126795776
 10.704491  41.252598    0.15    0.14      24 ...    69157   1590591    33 4805203678126855168
 10.704360  41.257473    0.19    0.19      13 ... 

In [ ]:
coord = SkyCoord(ra_deg, dec_deg, unit='deg', frame='galactic')
table = Irsa.query_region(coordinates=coord,
                          catalog='fp_psc', radius='0d2m0s')
print(table)

    ra        dec     err_maj err_min err_ang ... scan_key coadd_key coadd        htm20       
   deg        deg      arcsec  arcsec   deg   ...                                             
---------- ---------- ------- ------- ------- ... -------- --------- ----- -------------------
237.004404   2.751914    0.27    0.25     179 ...    53725   1235665   150 4803753526738309120
237.003187   2.737467    0.09    0.08       0 ...    53725   1235665   150 4803753526788968960
237.015384   2.732973    0.48    0.33      84 ...    53725   1235665   150 4803753526799783424
236.995817   2.709317    0.07    0.06      11 ...    53724   1235640   127 4803753526822579712


In [ ]:
# Option 2: Display available data by searching using coordinates of galaxy
ra = 121.1743
dec = -21.5733
coord = SkyCoord(ra, dec, unit='deg', frame='galactic')
table = Irsa.query_region(coordinates=coord,
                          catalog='fp_psc', radius='0d2m0s')
print(table)

    ra        dec     err_maj err_min err_ang ... scan_key coadd_key coadd        htm20       
   deg        deg      arcsec  arcsec   deg   ...                                             
---------- ---------- ------- ------- ------- ... -------- --------- ----- -------------------
 10.692216  41.260162    0.10    0.09      87 ...    69157   1590591    33 4805203678124326400
 10.700059  41.263481    0.31    0.30     155 ...    69157   1590591    33 4805203678125364736
 10.699131  41.263248    0.28    0.20      82 ...    69157   1590591    33 4805203678125474304
 10.697569  41.261272    0.11    0.10      90 ...    69157   1590591    33 4805203678125530624
 10.703106  41.252998    0.16    0.14      21 ...    69157   1590591    33 4805203678126695936
 10.703557  41.252811    0.14    0.13     111 ...    69157   1590591    33 4805203678126795776
 10.704491  41.252598    0.15    0.14      24 ...    69157   1590591    33 4805203678126855168
 10.704360  41.257473    0.19    0.19      13 ... 

In [ ]:
Irsa.list_collections()

collection
object
akari_allskymaps
blast
bolocam_gps
bolocam_lh
bolocam_planck_sz
champ
cosmos
euclid_ero
goals


In [ ]:
# Check available catalogs
catalogs = Irsa.list_catalogs()

# Check available catalogs
print(catalogs)

# Filter for Spitzer catalogs
spitzer_catalogs = {key: value for key, value in catalogs.items() if 'spitzer' in key}

# Display the Spitzer catalogs
print(spitzer_catalogs)

{'spitzer.fls_vla': 'Spitzer FLS VLA Image Metadata', 'spitzer.fidel_images': 'Far-Infrared Deep Extragalactic Legacy Survey (FIDEL) Images', 'spitzer.fidel_images_24um': 'Spitzer FIDEL 24um Image Metadata', 'spitzer.fidel_images_70um': 'Spitzer FIDEL 70um Image Metadata', 'spitzer.fidel_images_160um': 'Spitzer FIDEL 160um Image Metadata', 'spitzer.apoglimpse_0_6_images': 'Spitzer APOGlimpse Images', 'spitzer.apoglimpse_1_2_images': 'Spitzer APOGlimpse Images', 'spitzer.glimpsei_0_6': 'Spitzer GlimpseI 0.6 Image Metadata', 'spitzer.glimpsei_1_2': 'Spitzer GlimpseI 1.2 Image Metadata', 'spitzer.glimpseii_0_6': 'Spitzer GlimpseII 0.6 Image Metadata', 'spitzer.glimpseii_1_2': 'Spitzer GlimpseII 1.2 Image Metadata', 'spitzer.glimpseii_sub': 'Spitzer GlimpseII Sub Image Metadata', 'spitzer.glimpse3d_0_6': 'Spitzer Glimpse3d 0.6 Image Metadata', 'spitzer.glimpse3d_1_2': 'Spitzer Glimpse3d 1.2 Image Metadata', 'spitzer.glimpse360_0_6': 'Spitzer Glimpse360 0.6 Image Metadata', 'spitzer.glimpse

In [ ]:
# Convert Right Ascension (RA) and Declination (DEC) to decimal degree from Simbad
coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=('hourangle', 'deg'), frame='icrs')
ra_deg = round(coord.ra.deg, 2)
dec_deg = round(coord.dec.deg, 2)

print("RA in degrees:", ra_deg)
print("DEC in degrees:", dec_deg)

RA in degrees: 10.68
DEC in degrees: 41.27


In [ ]:
# Queries over a polygon
table = Irsa.query_region("m31", catalog="fp_psc", spatial="Polygon",
polygon=[coordinates.SkyCoord(ra=10.1, dec=10.1, unit=(u.deg, u.deg), frame='icrs'),
         coordinates.SkyCoord(ra=10.0, dec=10.1, unit=(u.deg, u.deg), frame='icrs'),
         coordinates.SkyCoord(ra=10.0, dec=10.0, unit=(u.deg, u.deg), frame='icrs')
        ])
print(table)

    ra        dec     err_maj err_min err_ang ... scan_key coadd_key coadd        htm20       
   deg        deg      arcsec  arcsec   deg   ...                                             
---------- ---------- ------- ------- ------- ... -------- --------- ----- -------------------
 10.015839  10.038061    0.09    0.06      90 ...    62740   1443005    91 4805087709670704640
 10.015696  10.099228    0.10    0.07      90 ...    62740   1443005    91 4805087709940635648
 10.011170  10.093903    0.23    0.21     167 ...    62740   1443005    91 4805087710032524288
 10.031016  10.063082    0.19    0.18     114 ...    62740   1443005    91 4805087710169327616
 10.036776  10.060278    0.11    0.06      90 ...    62740   1443005    91 4805087710175392768
 10.059964  10.085445    0.23    0.20      96 ...    62740   1443005    91 4805087710674674176
 10.005549  10.018401    0.16    0.14     108 ...    62740   1443005    91 4805087784811171840


In [ ]:
# Selecting columns
table = Irsa.query_region("m31", catalog="allwise_p3as_psd", spatial="Cone", columns="ra,dec,w1mpro")
print(table)

     ra         dec      w1mpro
    deg         deg       mag  
----------- ----------- -------
 10.6846947  41.2689392   5.819


In [ ]:
# Direct TAP query to the IRSA server
query = ("SELECT TOP 10 ra,dec,j_m,j_msigcom,h_m,h_msigcom,k_m,k_msigcom,ph_qual,cc_flg "
         "FROM fp_psc WHERE CONTAINS(POINT('ICRS',ra, dec), CIRCLE('ICRS',10.6846947,41.2689392,5.819))=1")
#          "FROM fp_psc WHERE CONTAINS(POINT('ICRS',ra, dec), CIRCLE('ICRS',202.48417,47.23056,0.4))=1")
results = Irsa.query_tap(query=query).to_qtable()
results

/usr/local/lib/python3.10/dist-packages/pyvo/dal/query.py:325: DALOverflowWarning: Partial result set. Potential causes MAXREC, async storage space, etc.
  warn("Partial result set. Potential causes MAXREC, async storage space, etc.",


ra,dec,j_m,j_msigcom,h_m,h_msigcom,k_m,k_msigcom,ph_qual,cc_flg
deg,deg,mag,mag,mag,mag,mag,mag,,
float64,float64,float32,float32,float32,float32,float32,float32,object,object
17.182749,43.699284,16.484,0.105,15.971,0.145,15.949,0.236,BBD,000
17.198810,43.716602,15.869,0.065,15.295,0.090,15.174,0.123,AAB,000
17.190077,43.718616,14.606,0.031,13.990,0.033,13.763,0.037,AAA,000
17.198377,43.714397,16.412,0.097,15.796,0.138,15.752,0.182,ABC,000
17.174449,43.714855,15.046,0.039,14.365,0.044,14.352,0.063,AAA,000
17.186766,43.715275,15.655,0.059,15.427,0.090,15.548,0.163,AAC,000
17.189046,43.706406,16.128,0.088,15.689,0.111,15.572,0.172,ABC,000
17.205421,43.752987,12.666,0.022,12.145,0.022,12.072,0.021,AAA,000


In [ ]:
# Reverse check if coordinate belong to correct object
coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=('hourangle', 'deg'), frame='icrs')
result = Simbad.query_region(coord, radius='0d0m5s')  # Search within a small radius (5 arcseconds)
if result:
    print(result['MAIN_ID'][0])  # Show the main name of the object
else:
    print("No object found in SIMBAD at these coordinates.")

M  31
